<div align='center'>

# 🎙️ Elbek Studiya — RVC Model Trainer
### Ovoz modeli yaratish • Gradio UI + ngrok

| | |
|---|---|
| **1-qadam** | Muhit o'rnatish (bir marta) |
| **2-qadam** | Gradio UI ishga tushirish + havola olish |
| **3-qadam** | Havolaga kiring → Ovoz yuklang → Train |
| **4-qadam** | ZIP avtomatik yuklab olinadi |

⚡ **GPU**: `Runtime → Change runtime type → T4 GPU`
</div>

In [ ]:
#@title ⚙️ 1-QADAM — Muhit o'rnatish (GPU kerak!) { display-mode: "form" }
#@markdown Birinchi marta ~5-7 daqiqa ketadi. **Faqat bir marta ishga tushiring.**

import os, subprocess, sys

def run(cmd, show=True):
    """Bash komandani ishga tushiradi."""
    if show:
        print(f'\033[36m$ {cmd[:80]}\033[0m')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout and show:
        print(r.stdout[-1500:])
    if r.returncode != 0 and r.stderr:
        print(f'\033[31m{r.stderr[-800:]}\033[0m')
    return r

# ── GPU tekshirish ──────────────────────────────────────────────────────────
print('\033[1;33m🔍 GPU tekshirilmoqda...\033[0m')
g = run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', show=False)
if g.returncode != 0:
    print('\033[1;31m❌ GPU topilmadi! Runtime → Change runtime type → T4 GPU\033[0m')
    raise SystemExit('GPU kerak!')
print(f'✅ GPU: {g.stdout.strip()}')

# ── RVC repozitoriyasini klonlash ───────────────────────────────────────────
REPO = '/content/RVC'
print('\n\033[1;33m📦 RVC yuklanmoqda...\033[0m')
if not os.path.exists(REPO):
    run(f'git clone --depth=1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI {REPO}')
else:
    print('  ⏭️  Repo mavjud')

os.chdir(REPO)

# ── ffmpeg ─────────────────────────────────────────────────────────────────
print('\n\033[1;33m🔧 ffmpeg o\'rnatilmoqda...\033[0m')
run('apt-get install -qq -y ffmpeg', show=False)
print('  ✅ ffmpeg')

# ── Python kutubxonalari ────────────────────────────────────────────────────
print('\n\033[1;33m📦 Kutubxonalar o\'rnatilmoqda...\033[0m')
run('pip install -q --upgrade pip setuptools wheel', show=False)
# requirements.txt ni o'rnatish
run('pip install -q -r requirements.txt', show=False)
# Extra kerakli paketlar
run('pip install -q pyngrok gradio==3.50.2', show=False)
print('  ✅ Kutubxonalar')

# ── Pretrained modellarni yuklab olish ──────────────────────────────────────
print('\n\033[1;33m📥 Pretrained modellar yuklanmoqda...\033[0m')

ASSETS = [
    # (URL, lokal yo'l)
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt',
     f'{REPO}/assets/hubert/hubert_base.pt'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt',
     f'{REPO}/assets/rmvpe/rmvpe.pt'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth',
     f'{REPO}/assets/pretrained_v2/f0G40k.pth'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth',
     f'{REPO}/assets/pretrained_v2/f0D40k.pth'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/G40k.pth',
     f'{REPO}/assets/pretrained_v2/G40k.pth'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/D40k.pth',
     f'{REPO}/assets/pretrained_v2/D40k.pth'),
]

for url, dest in ASSETS:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        print(f'  ⏭️  {os.path.basename(dest)} (mavjud)')
    else:
        r = run(f'wget -q --show-progress -c -O "{dest}" "{url}"', show=False)
        if r.returncode == 0:
            sz = os.path.getsize(dest) / 1024 / 1024
            print(f'  ✅ {os.path.basename(dest)} ({sz:.1f} MB)')
        else:
            print(f'  ❌ {os.path.basename(dest)} YUKLANMADI!')

# ── weights va indices papkalarini yaratish ─────────────────────────────────
for d in ['assets/weights', 'assets/indices']:
    os.makedirs(os.path.join(REPO, d), exist_ok=True)

print('\n\033[1;32m✅✅✅ MUHIT TAYYOR! 2-qadam ga o\'ting.\033[0m')

In [ ]:
#@title 🚀 2-QADAM — Gradio UI ishga tushirish { display-mode: "form" }
#@markdown ### 🔑 ngrok token (ixtiyoriy)
#@markdown Bepul token: https://dashboard.ngrok.com/get-started/your-authtoken
#@markdown Token bo'lmasa ham Gradio `.gradio.live` havolasi chiqadi.
NGROK_TOKEN = '' #@param {type:"string"}

import gradio as gr
import os, shutil, zipfile, glob, subprocess, re, threading
from pathlib import Path

REPO = '/content/RVC'
os.chdir(REPO)

# ── ngrok tunnel ────────────────────────────────────────────────────────────
if NGROK_TOKEN and NGROK_TOKEN.strip():
    try:
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = NGROK_TOKEN.strip()
        ngrok.kill()
        tunnel = ngrok.connect(7860, 'http')
        print(f'\033[1;32m🔗 ngrok tunnel: {tunnel.public_url}\033[0m')
    except Exception as e:
        print(f'⚠️  ngrok xato: {e}')

# ═══════════════════════════════════════════════════════════════════════════
#  YORDAMCHI FUNKSIYALAR
# ═══════════════════════════════════════════════════════════════════════════

def clean_name(s):
    return re.sub(r'[^\w\-]', '_', (s or '').strip())

def shell(cmd, cwd=REPO, timeout=3600):
    """Komanda ishga tushiradi, stdout+stderr qaytaradi."""
    r = subprocess.run(
        cmd, shell=True, cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        timeout=timeout
    )
    return (r.stdout or '')[-4000:], r.returncode

def find_outputs(name):
    """Tayyor .pth va .index fayllarini barcha mumkin joylarda qidiradi."""
    pth  = (
        glob.glob(f'{REPO}/assets/weights/{name}*.pth') +
        glob.glob(f'{REPO}/logs/{name}/*.pth')
    )
    idx  = (
        glob.glob(f'{REPO}/assets/indices/{name}*.index') +
        glob.glob(f'{REPO}/logs/{name}/*.index') +
        glob.glob(f'{REPO}/logs/{name}/added_*.index')
    )
    return list(set(pth)), list(set(idx))

def build_zip(name, pth_files, idx_files):
    out = f'/content/{name}_ElbekStudiya.zip'
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in pth_files + idx_files:
            if os.path.exists(f):
                z.write(f, os.path.basename(f))
    return out if os.path.getsize(out) > 100 else None

# ═══════════════════════════════════════════════════════════════════════════
#  ASOSIY O'QITISH FUNKSIYASI
# ═══════════════════════════════════════════════════════════════════════════

def train_model(
    raw_name, audio_files, total_epoch,
    sample_rate, pitch_algo, batch_size, save_every,
    progress=gr.Progress(track_tqdm=True)
):
    log = []
    def L(msg):
        log.append(str(msg))
        return '\n'.join(log)

    # ── Kirish tekshirish ───────────────────────────────────────────────
    if not raw_name or not raw_name.strip():
        return '❌ Model nomini kiriting!', None
    if not audio_files:
        return '❌ Ovoz faylini yuklang!', None

    name = clean_name(raw_name)
    sr   = sample_rate  # '40k' yoki '48k'
    # sample_rate dan pretrained fayl nomini aniqlash
    sr_num = sr.replace('k', '000')  # '40k' -> '40000'
    sr_key = sr  # '40k' yoki '48k'

    L('═' * 55)
    L(f'🎙️  Elbek Studiya — RVC Trainer')
    L('═' * 55)
    L(f'🏷️  Model     : {name}')
    L(f'🔢  Epoch     : {total_epoch}')
    L(f'📻  SR        : {sr}')
    L(f'🎵  Pitch     : {pitch_algo}')
    L(f'📦  Batch     : {batch_size}')
    L(f'📁  Fayllar   : {len(audio_files)} ta')
    L('─' * 55)

    try:
        os.chdir(REPO)

        # ── 1. Dataset tayyorlash ────────────────────────────────────────
        progress(0.03, desc='📁 Dataset tayyorlanmoqda...')
        ds = f'{REPO}/dataset/{name}'
        shutil.rmtree(ds, ignore_errors=True)
        os.makedirs(ds, exist_ok=True)
        for f in audio_files:
            shutil.copy(f.name, os.path.join(ds, os.path.basename(f.name)))
        L(f'\n✅ [1/5] Dataset: {len(audio_files)} fayl → {ds}')

        log_dir = f'{REPO}/logs/{name}'
        os.makedirs(log_dir, exist_ok=True)

        # ── 2. Preprocessing (slice + resample) ─────────────────────────
        progress(0.10, desc='🔊 Ovoz qayta ishlanmoqda...')
        L('\n⏳ [2/5] Ovoz qayta ishlanmoqda (slice + resample)...')
        out, rc = shell(
            f'python trainset_preprocess_pipeline_print.py '
            f'"{ds}" {sr_num} 2 "{log_dir}" False'
        )
        L(out[-2000:] or '(chiqish yo\'q)')
        if rc != 0:
            L(f'⚠️  Preprocessing xato (rc={rc}) — davom etamiz')
        else:
            L('✅ Preprocessing tayyor')

        # ── 3. Pitch extraction ──────────────────────────────────────────
        progress(0.25, desc=f'🎵 Pitch chiqarilmoqda ({pitch_algo})...')
        L(f'\n⏳ [3/5] Pitch chiqarilmoqda ({pitch_algo})...')
        out, rc = shell(
            f'python extract_f0_print.py "{log_dir}" 2 {pitch_algo}'
        )
        L(out[-2000:] or '(chiqish yo\'q)')
        if rc == 0:
            L('✅ Pitch chiqarildi')

        # ── 4. Feature extraction (HuBERT) ───────────────────────────────
        progress(0.42, desc='🧠 HuBERT xususiyatlari...')
        L('\n⏳ [4/5] HuBERT xususiyatlari chiqarilmoqda...')
        out, rc = shell(
            f'python extract_feature_print.py cuda:0 1 0 0 "{log_dir}" v2'
        )
        L(out[-2000:] or '(chiqish yo\'q)')
        if rc == 0:
            L('✅ Xususiyatlar chiqarildi')

        # ── 5. Model o'qitish ─────────────────────────────────────────────
        progress(0.52, desc=f'🏋️ O\'qitish ({total_epoch} epoch)...')
        L(f'\n⏳ [5a/5] Model o\'qitilmoqda ({total_epoch} epoch)...')
        L('⏳ Bu biroz vaqt olishi mumkin, sabr qiling...')

        # Pretrained fayllar yo'li
        pg = f'assets/pretrained_v2/f0G{sr_key}.pth'
        pd = f'assets/pretrained_v2/f0D{sr_key}.pth'
        # Agar fayl yo'q bo'lsa fallback
        if not os.path.exists(os.path.join(REPO, pg)):
            pg = 'assets/pretrained_v2/f0G40k.pth'
            pd = 'assets/pretrained_v2/f0D40k.pth'

        train_cmd = (
            f'python train_nsf_sim_cache_sid_load_pretrain.py '
            f'-e "{name}" '
            f'-sr {sr} '
            f'-f0 1 '
            f'-bs {batch_size} '
            f'-g 0 '
            f'-te {total_epoch} '
            f'-se {save_every} '
            f'-pg {pg} '
            f'-pd {pd} '
            f'-l 0 '
            f'-c 0 '
            f'-sw 1 '
            f'-v v2'
        )
        out, rc = shell(train_cmd, timeout=7200)
        L(out[-3000:] or '(chiqish yo\'q)')
        if rc == 0:
            L('✅ O\'qitish yakunlandi')
        else:
            L(f'⚠️  O\'qitish xato (rc={rc})')

        # ── 6. Index yaratish ─────────────────────────────────────────────
        progress(0.88, desc='📑 Index yaratilmoqda...')
        L('\n⏳ [5b/5] Index yaratilmoqda...')
        # Metod 1: train_index.py
        out, rc = shell(f'python train_index.py "{log_dir}" v2')
        if rc != 0:
            # Metod 2: infer/modules/train/train.py orqali
            out2, rc2 = shell(
                f'python -c "from infer.modules.train.train import *; '
                f'train_index(exp_dir1=\'{log_dir}\', version19=\'v2\')"'
            )
            L(out2[-1000:] or '(chiqish yo\'q)')
        else:
            L(out[-1000:] or '✅ Index yaratildi')

        # ── 7. Fayllarni topish va ZIP yaratish ───────────────────────────
        progress(0.95, desc='📦 ZIP yaratilmoqda...')
        pth_f, idx_f = find_outputs(name)

        L('\n🔍 Fayllar qidirilmoqda...')
        L(f'   PTH   : {[os.path.basename(f) for f in pth_f] or "topilmadi"}')
        L(f'   INDEX : {[os.path.basename(f) for f in idx_f] or "topilmadi"}')

        if not pth_f and not idx_f:
            # Logs papkasidagi hamma narsani ko'rsatish
            all_files = glob.glob(f'{REPO}/logs/{name}/**', recursive=True)
            L('\n📂 Logs papkasi tarkibi:')
            for ff in all_files[:30]:
                L(f'   {ff}')
            return '\n'.join(log), None

        zip_path = build_zip(name, pth_f, idx_f)
        if not zip_path:
            L('❌ ZIP yaratishda xato!')
            return '\n'.join(log), None

        L('\n' + '█'*55)
        L('🎉  O\'QITISH MUVAFFAQIYATLI YAKUNLANDI!')
        L('█'*55)
        L(f'📦  ZIP: {zip_path}')
        L(f'   .pth   → {[os.path.basename(f) for f in pth_f]}')
        L(f'   .index → {[os.path.basename(f) for f in idx_f]}')
        L('⬇️   Quyidagi yuklab olish tugmasini bosing!')

        progress(1.0, desc='🎉 Tayyor!')
        return '\n'.join(log), zip_path

    except subprocess.TimeoutExpired:
        L('\n⏱️  Timeout! O\'qitish juda uzoq ketdi. Epoch sonini kamaytiring.')
        return '\n'.join(log), None
    except Exception:
        import traceback
        L(f'\n💥 Xato:\n{traceback.format_exc()}')
        return '\n'.join(log), None

# ═══════════════════════════════════════════════════════════════════════════
#  GRADIO INTERFEYS
# ═══════════════════════════════════════════════════════════════════════════

CSS = """
body, .gradio-container {
    background: #0a0a12 !important;
    font-family: 'Segoe UI', system-ui, sans-serif !important;
}
.es-top {
    background: linear-gradient(135deg,#12001f 0%,#1a0030 50%,#0d0025 100%);
    border: 2px solid #c0392b;
    border-radius: 18px;
    padding: 32px 24px 24px;
    text-align: center;
    margin-bottom: 18px;
    box-shadow: 0 0 40px rgba(192,57,43,0.4), inset 0 1px 0 rgba(255,255,255,0.05);
}
.es-top h1 {
    color: #fff;
    font-size: 2.4em;
    font-weight: 800;
    margin: 0 0 6px;
    text-shadow: 0 0 20px rgba(192,57,43,0.8);
    letter-spacing: 1px;
}
.es-top .red  { color: #e74c3c; }
.es-top .sub  { color: #9b59b6; font-size:1em; margin:0; }
.es-top .chip {
    display:inline-block; background:#c0392b; color:#fff;
    border-radius:20px; padding:4px 14px; font-size:.78em;
    font-weight:700; margin-top:10px; letter-spacing:.5px;
}
.gr-button { border-radius: 10px !important; font-weight:700 !important; }
.gr-button.gr-button-primary {
    background: linear-gradient(135deg,#c0392b,#922b21) !important;
    border: none !important;
    font-size: 1.15em !important;
    padding: 15px !important;
    box-shadow: 0 4px 20px rgba(192,57,43,.5) !important;
    transition: all .25s !important;
}
.gr-button.gr-button-primary:hover {
    transform: translateY(-2px) scale(1.01) !important;
    box-shadow: 0 8px 30px rgba(192,57,43,.7) !important;
}
.gr-form, .gr-box, .gr-block { border-radius:12px !important; }
label, .label-wrap span {
    color:#dfe6e9 !important;
    font-weight:600 !important;
    font-size:.92em !important;
}
textarea, input[type=text] {
    background:#1a1a2e !important;
    color:#ecf0f1 !important;
    border-color:#34495e !important;
}
.footer {
    text-align:center; color:#636e72; font-size:.8em;
    padding:12px; border-top:1px solid #2d3436; margin-top:14px;
}
"""

TOP_HTML = """
<div class='es-top'>
  <h1>🎙️ <span class='red'>Elbek Studiya</span> RVC Model Trainer</h1>
  <p class='sub'>Ovoz yuklang · Model nomi kiriting · Train bosing · ZIP yuklab oling</p>
  <span class='chip'>⚡ Retrieval-based Voice Conversion v2</span>
</div>
"""

theme = gr.themes.Base(
    primary_hue=gr.themes.colors.red,
    secondary_hue=gr.themes.colors.purple,
    neutral_hue=gr.themes.colors.gray,
).set(
    body_background_fill='#0a0a12',
    block_background_fill='#13131f',
    block_border_color='#2d2d44',
    input_background_fill='#1a1a2e',
    button_primary_background_fill='#c0392b',
    button_primary_background_fill_hover='#e74c3c',
    button_primary_text_color='white',
)

with gr.Blocks(css=CSS, theme=theme, title='🎙️ Elbek Studiya RVC') as demo:

    gr.HTML(TOP_HTML)

    with gr.Row(equal_height=False):

        # ━━━━ CHAP: KIRISH ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        with gr.Column(scale=5, min_width=340):
            gr.Markdown('### 📋 Model sozlamalari')

            model_inp = gr.Textbox(
                label='🏷️ Model nomi',
                placeholder='masalan:  sarvar_ovozi   yoki   elbek_2024',
                max_lines=1,
                info='Faqat harflar, raqamlar, _ — bo\'sh joy bo\'lmaydi'
            )

            audio_inp = gr.File(
                label='🎤 Ovoz fayllari  (.wav / .mp3 / .flac)',
                file_count='multiple',
                file_types=['.wav','.mp3','.flac','.ogg','.m4a'],
            )

            with gr.Accordion('⚙️ Kengaytirilgan sozlamalar', open=True):
                epoch_inp = gr.Slider(
                    label='🔢 Epoch soni',
                    minimum=10, maximum=1000, value=100, step=10,
                    info='Tavsiya: test=100, normal=200, yuqori sifat=300-500'
                )
                with gr.Row():
                    sr_inp = gr.Radio(
                        label='📻 Sample Rate',
                        choices=['40k','48k'],
                        value='40k',
                        info='40k — standart, 48k — premium'
                    )
                    pitch_inp = gr.Dropdown(
                        label='🎵 Pitch algoritmi',
                        choices=['rmvpe','harvest','crepe','pm'],
                        value='rmvpe',
                        info='rmvpe = eng aniq'
                    )
                with gr.Row():
                    batch_inp = gr.Slider(
                        label='📦 Batch size',
                        minimum=1, maximum=24, value=4, step=1,
                        info='T4 GPU uchun 4-8'
                    )
                    save_inp = gr.Slider(
                        label='💾 Har necha epochda saqlash',
                        minimum=5, maximum=100, value=10, step=5
                    )

            train_btn = gr.Button(
                '🚀   O\'QITISHNI BOSHLASH',
                variant='primary', size='lg'
            )

            gr.Markdown("""
---
**📌 Qo'llanma:**  
① Model nomini kiriting &nbsp;②  Ovoz yuklang &nbsp;③  Epoch tanlang  
④ **Train** bosing &nbsp;⑤  ZIP yuklab oling  

**💡 Yaxshi natija uchun:** 10-20 daqiqa sifatli, shovqinsiz ovoz
""")

        # ━━━━ O'NG: NATIJA ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        with gr.Column(scale=5, min_width=340):
            gr.Markdown('### 📊 Jarayon va natija')

            log_out = gr.Textbox(
                label='📜 Jarayon jurnali',
                lines=26,
                interactive=False,
                show_copy_button=True,
                placeholder=(
                    'O\'qitish boshlanganidan keyin bu yerda ko\'rinadi:\n\n'
                    '  ══════════════════════════════\n'
                    '  🎙️  Elbek Studiya — RVC Trainer\n'
                    '  ══════════════════════════════\n'
                    '  🏷️  Model     : sizning_ismingiz\n'
                    '  🔢  Epoch     : 100\n'
                    '  ──────────────────────────────\n'
                    '  ✅ [1/5] Dataset tayyor\n'
                    '  ⏳ [2/5] Ovoz qayta ishlanmoqda...\n'
                    '  ⏳ [3/5] Pitch chiqarilmoqda...\n'
                    '  ⏳ [4/5] HuBERT xususiyatlari...\n'
                    '  ⏳ [5/5] Model o\'qitilmoqda...\n'
                    '  ████████████████████████████\n'
                    '  🎉  MUVAFFAQIYATLI YAKUNLANDI!'
                )
            )

            zip_out = gr.File(
                label='📦 Tayyor model — ZIP yuklab olish (.pth + .index)',
                interactive=False
            )

            gr.Markdown("""
---
**📦 ZIP tarkibi:**
```
model_nomi_ElbekStudiya.zip
├── model_nomi_e100.pth      ← asosiy model
└── added_IVF_model.index    ← qidiruv indeksi
```
**🔗 Bu modelni qayerlarda ishlatsa bo'ladi:**  
[RVC Simple (HuggingFace)](https://huggingface.co/spaces/r3gm/RVC_HF) •
[AICoverGen](https://github.com/SociallyIneptWeeb/AICoverGen)
""")

    gr.HTML("""
<div class='footer'>
  🎙️ <strong>Elbek Studiya RVC Trainer</strong> &nbsp;•&nbsp;
  Powered by <a href='https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI'
    style='color:#e74c3c;text-decoration:none'>RVC-Project</a> &nbsp;•&nbsp;
  UI: <a href='https://gradio.app' style='color:#9b59b6;text-decoration:none'>Gradio</a>
</div>
""")

    train_btn.click(
        fn=train_model,
        inputs=[model_inp, audio_inp, epoch_inp, sr_inp,
                pitch_inp, batch_inp, save_inp],
        outputs=[log_out, zip_out],
        show_progress=True
    )

# ─── Ishga tushirish ────────────────────────────────────────────────────────
print('\n\033[1;32m🚀 Gradio UI ishga tushmoqda...\033[0m')
print('\033[1;33m⏳ Bir daqiqa kuting, havola chiqadi...\033[0m\n')

demo.queue(max_size=3).launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=True,       # ← gradio.live havolasini yaratadi
    debug=False,
    show_error=True,
    quiet=False,
    inbrowser=False
)